### Manual inspection of hero prediction and matches
- Sampled 15% of hero test dataset stratified by mbfc
- Inspected judge matches

Conclusion: MOSTLY looks good to me

- Found a problem with ID 479 (wildfire as a hero). Incorrect label, correct 
prediction, incorrect match.

- ID 231 is debatable, but I still consider it a match given the text.

Details below


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from dotenv import load_dotenv
import dspy

load_dotenv()

openai_key = os.getenv(
        "OPENAI_API_KEY"
    )


/Users/catherine/Library/Caches/pypoetry/virtualenvs/afan-WLhS1US3-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from afan.dataset import load_and_preprocess_dataset

hero_test = load_and_preprocess_dataset(
    dataset_name="hero_test",
    data_dir="../data/test/",
    columns=["ID", "mbfc", "text", "entities"]
)

hero_test.head(2)

,ID,mbfc,text,entities
0,804,left_center_bias,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]"
1,366,right_bias,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ..."


In [4]:
hero_test.shape[0]

76

In [29]:
from sklearn.model_selection import train_test_split

# Calculate sample size (25% of total)
sample_size = int(0.25 * hero_test.shape[0])

# Get stratified sample
_, sampled_hero = train_test_split(
    hero_test,
    test_size=0.15,
    stratify=hero_test['mbfc'],
    random_state=42
)

# Display result
print(f"Original size: {hero_test.shape[0]}")
print(f"Sample size: {sampled_hero.shape[0]}")
print("\nMBFC distribution in sample:")
print(sampled_hero['mbfc'].value_counts())

Original size: 76
Sample size: 12

MBFC distribution in sample:
mbfc
left_bias              4
right_bias             3
left_center_bias       3
questionable_source    2
Name: count, dtype: int64


In [30]:
sampled_hero

,ID,mbfc,text,entities
57,78,right_bias,First meat grown in space lab 248 miles from Earth... Lab-grown meat has been successfully cultured in space for the first time. The Israeli food technology startup Aleph Farms grew the meat on th...,"[Lab-grown meat, The Israeli food technology startup Aleph Farms]"
43,690,left_bias,Trump's war on climate policy is also a war on public health Environmental Protection Agency chief Scott Pruitt is expected to sign a proposed rule on Tuesday that would roll back a key piece of P...,"[Obama, Clean Power Plan]"
18,697,left_bias,"California governor declares end to drought emergency LOS ANGELES (Reuters) – One of the worst droughts in California history has officially ended, Governor Jerry Brown declared, but not before it...",[Governor Jerry Brown]
46,847,questionable_source,"California districts sue US govt for $1.4bn over water contamination from military base Two water districts in Sacramento, California are suing the US government for nearly one and a half billion ...",[California water districts]
54,339,right_bias,"All These Climate Change Lawsuits Will Be Thrown Out Last week, a federal judge dismissed New York City's climate change lawsuit against five major oil companies. Last month, another federal judge...","[U.S. Courts, US District Judge William Alsup]"
36,44,right_bias,"Kids Around The World Are Using Climate Change As An Excuse To Skip School Young students across the world plan to skip class on Friday, claiming that they will devote the day to protesting man-ma...",[Climate activists]
71,789,left_center_bias,"EPA wipes its climate change site as protesters march in Washington The US Environmental Protection Agency's main climate change website is ""undergoing changes"" to better reflect ""the agency's new...","[Climate activists, senator]"
16,838,questionable_source,"250,000 Americans, 500 groups reject genetically-modified eucalyptus trees Over 250,000 Americans and 500 organizations submitted comments rejecting the US Department of Agriculture's (USDA) propo...",[Anne Petermann executive director at the Global Justice Ecology Project]
29,177,left_center_bias,"The sea-cooled eco-resort that's nearly mosquito-free The Brando is one of the most luxurious eco-resorts on the planet, nestling on an atoll in the middle of the Pacific Ocean. It's the last plac...",[The Brando]
60,231,left_center_bias,"Global Youth Climate Strike Expected To Draw Large Crowds Spurred by what they see as a sluggish, ineffectual response to the existential threat of global warming, student activists from around th...",[youth climate activism]


# Predict: gpt-4o-mini

In [31]:
prediction_lm = dspy.LM("openai/gpt-4o-mini", api_key=openai_key)

dspy.configure(lm=prediction_lm)

In [32]:
from afan.prompts.signatures import NarrativeArcSignature

predictor = dspy.Predict(NarrativeArcSignature)

In [33]:
from afan.utils import predict

hero_preds = predict(
    df=sampled_hero,
    predictor=predictor,
    entity="hero"
    )

hero_preds.head(2)

,ID,mbfc,text,entities,predicted_entity
57,78,right_bias,First meat grown in space lab 248 miles from Earth... Lab-grown meat has been successfully cultured in space for the first time. The Israeli food technology startup Aleph Farms grew the meat on th...,"[Lab-grown meat, The Israeli food technology startup Aleph Farms]","Aleph Farms, the Israeli food technology startup."
43,690,left_bias,Trump's war on climate policy is also a war on public health Environmental Protection Agency chief Scott Pruitt is expected to sign a proposed rule on Tuesday that would roll back a key piece of P...,"[Obama, Clean Power Plan]",The Clean Power Plan and public health advocates.


# Judge

In [34]:
judge_lm = dspy.LM("gpt-4.1-mini", api_key=openai_key)

dspy.configure(lm=judge_lm)

In [35]:
from afan.prompts.judges import EntitiesMatchFewShot

judge_few_shot = dspy.Predict(EntitiesMatchFewShot)

In [36]:
from afan.utils import judge

hero_judged = judge(
    df=hero_preds,
    judge_match=judge_few_shot
)

Accuracy: 75.00%


In [43]:
import pandas as pd

pd.set_option('display.max_colwidth', 200)

cols_to_compare = ["ID", "mbfc", "entities", "predicted_entity", "judge_match"]

In [38]:
# A wildfire as a hero? That's a problem. ID 479


hero_test[hero_test.ID == 479].text.values[0]

'California wildfires: Key questions answered In the last year alone, California has seen catastrophic wildfires burn thousands of acres. On Tuesday, the ongoing Camp Fire in Northern California became the state\'s deadliest blaze in history. How do these deadly fires start, and why is California so susceptible?Officials define wildfires, or wildland fires, as any fire occurring on undeveloped land. Forest fires are uncontrolled wildfires burning in lands covered at least in part by timber or flammable vegetation. In nature, two things can spark wildfires: lava and lightning. According to the National Park Service, 90% of wildfires in the US are caused by humans - whether by unattended campfires, burning debris, cigarettes, or arson. About 2% of California\'s 2,816 wildfires in 2016 were caused by lightning, according to data from the California Department of Forestry and Fire Protection (Cal Fire). The majority could be attributed to humans, though only 8% were caused by arson. Accord

In [ ]:
hero_judged[hero_judged.ID == 479]

In [ ]:
# debatable: ID 231. But I still think it's correct given the text


hero_test[hero_test.ID == 231].text.values[0]

'Global Youth Climate Strike Expected To Draw Large Crowds Spurred by what they see as a sluggish, ineffectual response to the existential threat of global warming, student activists from around the world plan to skip school today for what organizers call a Global Climate Strike. The focus for young activists is the United Nations Climate Action Summit taking place Monday in New York City. The strike\'s figurehead is 16-year-old Swedish activist Greta Thunberg, who traveled from Sweden to New York on an emission-free sailboat. A little over a year ago Thunberg began her school strike for the climate, by herself, outside the Swedish Parliament. Support for a school climate strike has since spread across the globe. In the last year Thunberg has been nominated for the Nobel Peace Prize by Norwegian lawmakers. She\'s also met with Pope Francis and lawmakers in several countries." We are currently on track for a world that could displace billions of people from their homes," Thunberg warned

In [46]:
hero_judged[cols_to_compare]

,ID,mbfc,entities,predicted_entity,judge_match
57,78,right_bias,"[Lab-grown meat, The Israeli food technology startup Aleph Farms]","Aleph Farms, the Israeli food technology startup.",True
43,690,left_bias,"[Obama, Clean Power Plan]",The Clean Power Plan and public health advocates.,True
18,697,left_bias,[Governor Jerry Brown],California residents who adapted their water usage to combat the drought.,False
46,847,questionable_source,[California water districts],Sacramento Suburban Water District and Rio Linda Elverta Community Water District,True
54,339,right_bias,"[U.S. Courts, US District Judge William Alsup]",The federal judges who dismissed the lawsuits.,True
36,44,right_bias,[Climate activists],Youth activists,True
71,789,left_center_bias,"[Climate activists, senator]","The protesters and climate activists, including notable figures like Al Gore and Leonardo DiCaprio, advocating for climate justice.",True
16,838,questionable_source,[Anne Petermann executive director at the Global Justice Ecology Project],Activists and environmental organizations advocating for ecological safety.,False
29,177,left_center_bias,[The Brando],The Brando eco-resort.,True
60,231,left_center_bias,[youth climate activism],Greta Thunberg,True


In [45]:
hero_judged[cols_to_compare][hero_judged.judge_match==False]

,ID,mbfc,entities,predicted_entity,judge_match
18,697,left_bias,[Governor Jerry Brown],California residents who adapted their water usage to combat the drought.,False
16,838,questionable_source,[Anne Petermann executive director at the Global Justice Ecology Project],Activists and environmental organizations advocating for ecological safety.,False
58,674,left_bias,[Boasberg],Standing Rock Sioux tribe,False
